# KairoDrishti LLVIP YOLO Training

Attach the private LLVIP dataset, enable a GPU accelerator, then run all cells. The notebook creates a deterministic 85/15 split and saves the trained model and metrics under `/kaggle/working/artifacts`.

In [ ]:
!pip install -q ultralytics
!nvidia-smi

In [ ]:
from pathlib import Path
import json, random, shutil, zipfile
from xml.etree import ElementTree

SEED = 42
VAL_FRACTION = 0.15
WORK = Path('/kaggle/working')
DATASET = WORK / 'llvip_yolo'
RAW = WORK / 'llvip_raw_yolo'

# Locate either the uploaded raw LLVIP.zip or an already prepared YOLO dataset.
zips = list(Path('/kaggle/input').rglob('LLVIP.zip'))
prepared = next((p for p in Path('/kaggle/input').rglob('dataset.yaml')), None)
if zips:
    archive = zips[0]
    RAW.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        names = set(bundle.namelist())
        for annotation_name in sorted(n for n in names if n.startswith('LLVIP/Annotations/') and n.endswith('.xml')):
            frame_id = Path(annotation_name).stem
            image_name = f'LLVIP/visible/train/{frame_id}.jpg'
            if image_name not in names:
                continue
            root = ElementTree.fromstring(bundle.read(annotation_name))
            size = root.find('size')
            width, height = int(size.findtext('width')), int(size.findtext('height'))
            labels = []
            for obj in root.findall('object'):
                if obj.findtext('name', '').lower() != 'person':
                    continue
                box = obj.find('bndbox')
                x0, y0 = float(box.findtext('xmin')), float(box.findtext('ymin'))
                x1, y1 = float(box.findtext('xmax')), float(box.findtext('ymax'))
                labels.append(f'0 {((x0+x1)/2)/width:.6f} {((y0+y1)/2)/height:.6f} {(x1-x0)/width:.6f} {(y1-y0)/height:.6f}')
            image_out = RAW / 'images' / 'train' / f'{frame_id}.jpg'
            label_out = RAW / 'labels' / 'train' / f'{frame_id}.txt'
            image_out.parent.mkdir(parents=True, exist_ok=True)
            label_out.parent.mkdir(parents=True, exist_ok=True)
            image_out.write_bytes(bundle.read(image_name))
            label_out.write_text('\n'.join(labels), encoding='utf-8')
    source = RAW
else:
    if prepared is None:
        raise FileNotFoundError('Attach LLVIP.zip or a prepared YOLO dataset')
    source = prepared.parent

pairs = []
for image in sorted((source / 'images' / 'train').glob('*.jpg')):
    label = source / 'labels' / 'train' / f'{image.stem}.txt'
    if label.exists():
        pairs.append((image, label))
random.Random(SEED).shuffle(pairs)
val_count = max(1, round(len(pairs) * VAL_FRACTION))
for split in ('train', 'val'):
    (DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)
for index, (image, label) in enumerate(pairs):
    split = 'val' if index < val_count else 'train'
    shutil.copy2(image, DATASET / 'images' / split / image.name)
    shutil.copy2(label, DATASET / 'labels' / split / label.name)
(DATASET / 'dataset.yaml').write_text('path: .\ntrain: images/train\nval: images/val\nnames:\n  0: person\n', encoding='utf-8')
print({'train': len(pairs) - val_count, 'val': val_count, 'dataset': DATASET})

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=str(DATASET / 'dataset.yaml'),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    cache=False,
    project=str(WORK / 'runs'),
    name='kairodristi-llvip',
)

In [ ]:
ARTIFACTS = WORK / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
RUN = WORK / 'runs' / 'kairodristi-llvip'
for path in [RUN / 'weights' / 'best.pt', RUN / 'weights' / 'last.pt', RUN / 'results.csv', RUN / 'results.png', RUN / 'confusion_matrix.png', RUN / 'args.yaml']:
    if path.exists():
        shutil.copy2(path, ARTIFACTS / path.name)
shutil.make_archive(str(WORK / 'kairodristi-llvip-artifacts'), 'zip', ARTIFACTS)
print('Download /kaggle/working/kairodristi-llvip-artifacts.zip')
print(sorted(p.name for p in ARTIFACTS.iterdir()))